# Task 4: LoRA Fine-Tuning — Granulometry Classification

Fine-tune Qwen2.5-VL-3B on 18 training images (2 per class) to classify concrete aggregate.

Two approaches:
- **Approach A**: Standard LoRA — direct fine-tuning on 18 labeled examples
- **Approach B**: SEAL-inspired LoRA — frontier model generates augmented training data (~150 examples)

Baseline (Task 3): ~36% size, ~34% grading (near random chance).

## Setup

In [ ]:
import os, json, re, time, torch, gc, random, base64
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')


## Config

In [ ]:
# Paths
TRAIN_DIR = '../../datasets/granulometry/train'
TEST_DIR = '../../datasets/granulometry/test'
TRAIN_MANIFEST = '../../datasets/granulometry/train_manifest.json'
TEST_MANIFEST = '../../datasets/granulometry/test_manifest.json'
REF_IMAGE_PATH = '../../task3-benchmarking/granulometry/examples_classification_data.png'

# Model
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
ORIGINAL_GSD = 8.0
MAX_DIM = 1500

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# Training
EPOCHS = 20
LR = 2e-5
BATCH_SIZE = 1
GRAD_ACCUM = 4
IMAGES_PER_CLASS = 2
SEED = 42

GRADING_DEFS = '''Grading (DIN 1045) describes particle size distribution:
- Coarse (A): most particles similar size, close to max. Uniform texture.
- Medium (B): moderate mix of large and small.
- Fine (C): wide range of sizes. Small particles fill gaps. Dense, packed texture.'''

EVAL_PROMPT = '''Classify this concrete aggregate photo. GSD = {gsd:.1f} px/mm.
Grading describes size distribution: coarse = uniform, mostly large; medium = moderate mix; fine = many small particles filling gaps, dense and packed.
Respond with ONLY JSON: {{"max_particle_size_mm": <8|16|32>, "grading": "<coarse|medium|fine>"}}'''


## Step 1: Select Training Images (18 total, 2 per class)

In [ ]:
random.seed(SEED)
with open(TRAIN_MANIFEST) as f:
    train_manifest = json.load(f)

by_class = defaultdict(list)
for e in train_manifest:
    by_class[e['class']].append(e)

selected = []
for cls in sorted(by_class):
    random.shuffle(by_class[cls])
    picked = by_class[cls][:IMAGES_PER_CLASS]
    selected.extend(picked)
    print(f'  {cls}: {[e["image"] for e in picked]}')

print(f'\nTotal training images: {len(selected)}')

# Preview
fig, axes = plt.subplots(2, 9, figsize=(27, 6))
for i, entry in enumerate(selected):
    img = Image.open(os.path.join(TRAIN_DIR, entry['image'])).convert('RGB')
    img.thumbnail((300, 300))
    axes[i//9][i%9].imshow(img)
    axes[i//9][i%9].set_title(f"{entry['class']}\n{entry['grading']},{entry['max_particle_size_mm']}mm", fontsize=8)
    axes[i//9][i%9].axis('off')
    img.close()
plt.suptitle('18 Training Images (2 per class)', fontsize=14)
plt.tight_layout(); plt.show()


## Step 2A: Prepare Direct Training Data (Approach A)

In [ ]:
DIRECT_PROMPT = (
    'Classify this concrete aggregate photo. GSD = 8.0 px/mm.\n'
    'Grading describes size distribution: coarse = uniform, mostly large; '
    'medium = moderate mix; fine = many small particles filling gaps, dense and packed.\n'
    'Respond with ONLY JSON: {"max_particle_size_mm": <8|16|32>, "grading": "<coarse|medium|fine>"}'
)

direct_data = []
for entry in selected:
    img_path = os.path.join(TRAIN_DIR, entry['image'])
    gt = json.dumps({'max_particle_size_mm': entry['max_particle_size_mm'], 'grading': entry['grading']})
    direct_data.append({
        'messages': [
            {'role': 'user', 'content': [{'type': 'image', 'image': img_path}, {'type': 'text', 'text': DIRECT_PROMPT}]},
            {'role': 'assistant', 'content': gt},
        ]
    })

with open('training_data_direct.jsonl', 'w') as f:
    for r in direct_data:
        f.write(json.dumps(r) + '\n')

print(f'Approach A: {len(direct_data)} direct training examples saved to training_data_direct.jsonl')


## Step 2B: Generate Augmented Training Data (Approach B — SEAL-inspired)

Uses Claude Opus 4.6 or GPT-5 to generate ~8 training variations per image.
Set your API key before running.

In [ ]:
# Set your API key
# os.environ['ANTHROPIC_API_KEY'] = 'your-key'  # for Claude
# os.environ['OPENAI_API_KEY'] = 'your-key'     # for GPT-5

PROVIDER = 'anthropic'  # or 'openai'
CLAUDE_MODEL = 'claude-sonnet-4-20250514'
GPT_MODEL = 'gpt-5'

AUG_PROMPT = '''You are helping create training data for a vision model that classifies concrete aggregate.

This image is class {cls} with:
- max_particle_size_mm: {size}
- grading: {grading}

{grading_defs}

GSD = 8.0 pixels per mm at original resolution.

Generate exactly 8 training examples as a JSON array. Each has "prompt" and "response".
Vary the styles:
1. Direct JSON (short prompt, JSON response)
2. Direct JSON with GSD in prompt
3. Chain-of-thought: reasoning then JSON
4. Visual description: describe particles, conclude with classification
5. Contrastive: explain why this grading, not the others
6. Size-focused: estimate sizes, conclude with max size
7. Grading-focused: describe distribution pattern
8. Minimal: very short prompt, just JSON response

All responses must include correct classification: max_particle_size_mm={size}, grading="{grading}".
Return ONLY the JSON array.'''

def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def call_frontier(image_b64, prompt):
    if PROVIDER == 'anthropic':
        import anthropic
        client = anthropic.Anthropic()
        msg = client.messages.create(
            model=CLAUDE_MODEL, max_tokens=4096,
            messages=[{'role': 'user', 'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': image_b64}},
                {'type': 'text', 'text': prompt},
            ]}],
        )
        return msg.content[0].text
    else:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=GPT_MODEL, max_tokens=4096,
            messages=[{'role': 'user', 'content': [
                {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{image_b64}'}},
                {'type': 'text', 'text': prompt},
            ]}],
        )
        return resp.choices[0].message.content

print(f'Provider: {PROVIDER}')
print('Ready to generate. Run the next cell to start.')


In [ ]:
# Generate augmented data — this calls the frontier model API
import time as _time

augmented_data = []
for entry in selected:
    img_path = os.path.join(TRAIN_DIR, entry['image'])
    if not os.path.exists(img_path):
        print(f'  SKIP: {img_path} not found'); continue

    prompt = AUG_PROMPT.format(cls=entry['class'], size=entry['max_particle_size_mm'],
                               grading=entry['grading'], grading_defs=GRADING_DEFS)
    print(f'  {entry["class"]}: {entry["image"]}...', end=' ', flush=True)

    try:
        raw = call_frontier(encode_image(img_path), prompt)
        # Parse JSON array
        raw_clean = re.sub(r'```json\s*', '', raw)
        raw_clean = re.sub(r'```\s*', '', raw_clean).strip()
        examples = json.loads(raw_clean) if raw_clean.startswith('[') else None
        if not examples:
            m = re.search(r'\[.*\]', raw_clean, re.DOTALL)
            examples = json.loads(m.group()) if m else None

        if examples and isinstance(examples, list):
            for ex in examples:
                augmented_data.append({
                    'messages': [
                        {'role': 'user', 'content': [
                            {'type': 'image', 'image': img_path},
                            {'type': 'text', 'text': ex['prompt']},
                        ]},
                        {'role': 'assistant', 'content': ex['response']},
                    ]
                })
            print(f'{len(examples)} examples')
        else:
            print('PARSE FAILED')
    except Exception as e:
        print(f'ERROR: {e}')
    _time.sleep(1)

with open('training_data_augmented.jsonl', 'w') as f:
    for r in augmented_data:
        f.write(json.dumps(r) + '\n')

print(f'\nApproach B: {len(augmented_data)} augmented examples saved to training_data_augmented.jsonl')


## Step 3: Load Model with QLoRA

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto', torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, task_type='CAUSAL_LM', bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.1f} GB')


## Step 4A: Train — Standard LoRA (18 examples)

In [ ]:
# TODO: Implement training loop
# The multimodal data collator for Qwen2.5-VL requires specific handling.
# Options:
#   1. trl.SFTTrainer with custom collator
#   2. Manual loop: processor -> model.forward() -> loss.backward()
#
# Check installed versions:
import transformers, peft, trl
print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
print(f'trl: {trl.__version__}')
print(f'\nTraining data: {len(direct_data)} examples')
print(f'Config: epochs={EPOCHS}, lr={LR}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}')
print(f'Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'Total steps: ~{len(direct_data) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)}')


## Step 4B: Train — SEAL-Augmented LoRA (~150 examples)

In [ ]:
# TODO: Same training loop as 4A but with augmented data
# Load augmented data:
with open('training_data_augmented.jsonl') as f:
    aug_data = [json.loads(line) for line in f]
print(f'Augmented training data: {len(aug_data)} examples')
print(f'Total steps: ~{len(aug_data) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)}')


## Step 5: Evaluate on Test Set

In [ ]:
def parse_response(raw):
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```\s*', '', raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    size_m = re.search(r'"max_particle_size_mm"\s*:\s*(\d+)', raw)
    grad_m = re.search(r'"grading"\s*:\s*"(\w+)"', raw)
    if size_m and grad_m:
        return {'max_particle_size_mm': int(size_m.group(1)), 'grading': grad_m.group(1)}
    return None

def run_eval(model, processor, manifest):
    results = []
    c_size = 0; c_grading = 0; v_json = 0; t_time = 0
    for i, entry in enumerate(manifest):
        img_path = os.path.join(TEST_DIR, entry['image'])
        if not os.path.exists(img_path): continue
        image = Image.open(img_path).convert('RGB')
        scale = min(MAX_DIM / max(image.size), 1.0)
        if scale < 1.0:
            image = image.resize((int(image.width*scale), int(image.height*scale)), Image.Resampling.LANCZOS)
        actual_gsd = ORIGINAL_GSD * scale
        msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':EVAL_PROMPT.format(gsd=actual_gsd)}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
        t = time.time()
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=128, temperature=0.7, do_sample=True)
        elapsed = time.time() - t; t_time += elapsed
        out = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
        del inputs, ids; image.close(); torch.cuda.empty_cache()
        parsed = parse_response(out)
        gt_s = entry['max_particle_size_mm']; gt_g = entry['grading']
        s_ok = False; g_ok = False
        if parsed:
            v_json += 1
            ps = parsed.get('max_particle_size_mm')
            if isinstance(ps, str): ps = int(ps) if ps.isdigit() else None
            if ps == gt_s: s_ok = True; c_size += 1
            if parsed.get('grading','').lower() == gt_g: g_ok = True; c_grading += 1
        results.append({'image': entry['image'], 'class': entry['class'], 'gt_size': gt_s, 'gt_grading': gt_g,
            'predicted': parsed, 'raw': out, 'size_correct': s_ok, 'grading_correct': g_ok,
            'valid_json': parsed is not None, 'time_s': round(elapsed, 2)})
        if (i+1) % 20 == 0:
            n = i+1
            print(f'  [{n}/{len(manifest)}] Size: {c_size}/{n} ({c_size/n*100:.0f}%) | Grading: {c_grading}/{n} ({c_grading/n*100:.0f}%)')
    return results, c_size, c_grading, v_json, t_time

with open(TEST_MANIFEST) as f:
    test_manifest = json.load(f)
print(f'Test set: {len(test_manifest)} images')


In [ ]:
# Evaluate (run after training is complete)
# results_a, cs_a, cg_a, vj_a, tt_a = run_eval(model_a, processor, test_manifest)
# results_b, cs_b, cg_b, vj_b, tt_b = run_eval(model_b, processor, test_manifest)
print('Run evaluation after training is complete.')


## Step 6: Compare All Results

In [ ]:
def summarize(label, results, c_size, c_grading, v_json, t_time):
    n = len(results)
    both = sum(1 for r in results if r['size_correct'] and r['grading_correct'])
    return {'label': label, 'n': n, 'json': round(v_json/n*100,1), 'size': round(c_size/n*100,1),
            'grading': round(c_grading/n*100,1), 'both': round(both/n*100,1), 'time': round(t_time/n,2)}

# Load Task 3 baselines
baselines = {}
for mode in ['zero-shot', 'few-shot']:
    path = f'../../task3-benchmarking/granulometry/benchmark_results_{mode}.json'
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        baselines[mode] = d

print(f'{"Method":<30} {"JSON":>6} {"Size":>7} {"Grade":>7} {"Both":>7} {"Time":>6}')
print('-' * 65)
for mode, d in baselines.items():
    print(f'Baseline ({mode}){" "*(18-len(mode))} {d["json_validity_pct"]:>5.0f}% {d["size_accuracy_pct"]:>6.1f}% '
          f'{d["grading_accuracy_pct"]:>6.1f}% {d["both_correct_pct"]:>6.1f}% {d["avg_inference_time_s"]:>5.1f}s')
# TODO: Add LoRA Direct and LoRA Augmented rows after training
print('\n[Add LoRA results after training]')


## Visualization

In [ ]:
# TODO: Side-by-side bar charts comparing:
# - Baseline zero-shot vs few-shot vs LoRA Direct vs LoRA Augmented
# - Per-class breakdown for each method
# Run after evaluation is complete.
print('Run visualization after evaluation.')


## Save Results

In [ ]:
# TODO: Save results after evaluation
# for label, summary, results in [('direct', sum_a, results_a), ('augmented', sum_b, results_b)]:
#     fname = f'results_{label}.json'
#     with open(fname, 'w') as f:
#         json.dump({...}, f, indent=2)
#     print(f'Saved {fname}')
print('Save results after evaluation.')
